In [2]:
import json
import pandas as pd

with open('data.geojson', 'r') as f:
    geojson = json.load(f)

rows = []
for feature in geojson['features']:
    coords = feature['geometry']['coordinates']
    props = feature['properties']
    rows.append({
        'id': feature['id'],
        'brand': props.get('brand'),
        'lat': coords[1],
        'lon': coords[0],
        'address': props.get('addr:street', 'N/A')
    })

df = pd.DataFrame(rows)

print(f"Total minimarket: {len(df)}")
print(f"Indomaret: {len(df[df['brand'] == 'Indomaret'])}")
print(f"Alfamart: {len(df[df['brand'] == 'Alfamart'])}")
print()
print(df.head())

Total minimarket: 96
Indomaret: 63
Alfamart: 33

                id      brand       lat         lon address
0  node/1359471683  Indomaret -7.314559  112.789765     N/A
1  node/2989728339  Indomaret -7.311784  112.746417     N/A
2  node/2989728395  Indomaret -7.308946  112.744605     N/A
3  node/3067860977   Alfamart -7.301474  112.775559     N/A
4  node/3121609244  Indomaret -7.268070  112.796515     N/A


In [3]:
depot = {
    'id': 'depot',
    'brand': 'DEPOT',
    'lat': -7.2372,
    'lon': 112.6878,
    'address': 'Margomulyo Industrial Area'
}

depot_df = pd.DataFrame([depot])
df_full = pd.concat([depot_df, df], ignore_index=True)

print(df_full.head(3))
print(f"\nTotal nodes (depot + customers): {len(df_full)}")

                id      brand       lat         lon  \
0            depot      DEPOT -7.237200  112.687800   
1  node/1359471683  Indomaret -7.314559  112.789765   
2  node/2989728339  Indomaret -7.311784  112.746417   

                      address  
0  Margomulyo Industrial Area  
1                         N/A  
2                         N/A  

Total nodes (depot + customers): 97


In [4]:
pip install folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import folium

m = folium.Map(location=[-7.28, 112.73], zoom_start=12)

# Depot Plotting
folium.Marker(
    location=[depot['lat'], depot['lon']],
    popup='DEPOT',
    icon=folium.Icon(color='red', icon='home')
).add_to(m)

# Minimarkets Plotting
for _, row in df.iterrows():
    color = 'blue' if row['brand'] == 'Indomaret' else 'green'
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color=color,
        fill=True,
        popup=row['brand']
    ).add_to(m)

m.save('minimarket_surabaya.html')
print("Map saved!")

Map saved!


![alt text](image.png)

In [6]:
import numpy as np

# Assign demand
df['demand'] = 1

NUM_VEHICLES = 10
VEHICLE_CAPACITY = 10
DEPOT_INDEX = 0

print(df_full.head())
print(f"\nTotal demand: {df['demand'].sum()}")
print(f"Vehicles: {NUM_VEHICLES}, Capacity each: {VEHICLE_CAPACITY}")
print(f"Max total capacity: {NUM_VEHICLES * VEHICLE_CAPACITY}")

                id      brand       lat         lon  \
0            depot      DEPOT -7.237200  112.687800   
1  node/1359471683  Indomaret -7.314559  112.789765   
2  node/2989728339  Indomaret -7.311784  112.746417   
3  node/2989728395  Indomaret -7.308946  112.744605   
4  node/3067860977   Alfamart -7.301474  112.775559   

                      address  
0  Margomulyo Industrial Area  
1                         N/A  
2                         N/A  
3                         N/A  
4                         N/A  

Total demand: 96
Vehicles: 10, Capacity each: 10
Max total capacity: 100


In [7]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c

# Test: Depot's distance to the nearest minimarket
depot_row = df_full.iloc[0]
toko_pertama = df_full.iloc[1]

jarak = haversine(depot_row['lat'], depot_row['lon'],
                  toko_pertama['lat'], toko_pertama['lon'])
print(f"Jarak depot ke toko pertama: {jarak:.2f} km")

Jarak depot ke toko pertama: 14.16 km


In [8]:
import numpy as np

all_points = df_full[['lat', 'lon']].values
n = len(all_points)

distance_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        if i != j:
            distance_matrix[i][j] = haversine(
                all_points[i][0], all_points[i][1],
                all_points[j][0], all_points[j][1]
            )

print(f"Shape matrix: {distance_matrix.shape}")
print(f"\nContoh jarak depot (index 0) ke 5 toko pertama:")
print(distance_matrix[0][:6].round(2))

Shape matrix: (97, 97)

Contoh jarak depot (index 0) ke 5 toko pertama:
[ 0.   14.16 10.52 10.14 12.03 12.47]


In [9]:
unvisited = list(range(1, len(df_full)))
routes = []

while unvisited:
    route = []
    load = 0
    pos = 0

    while True:
        if not unvisited:
            break

        nearest = min(unvisited, key=lambda x: distance_matrix[pos][x])

        if load + 1 <= VEHICLE_CAPACITY:
            route.append(nearest)
            unvisited.remove(nearest)
            load += 1
            pos = nearest
        else:
            break

    routes.append(route)

print(f"Jumlah rute: {len(routes)}")
for i, r in enumerate(routes):
    print(f"Truk {i+1}: {len(r)} toko")

Jumlah rute: 10
Truk 1: 10 toko
Truk 2: 10 toko
Truk 3: 10 toko
Truk 4: 10 toko
Truk 5: 10 toko
Truk 6: 10 toko
Truk 7: 10 toko
Truk 8: 10 toko
Truk 9: 10 toko
Truk 10: 6 toko


In [10]:
def hitung_total_jarak(routes, distance_matrix):
    total = 0
    for route in routes:
        pos = 0
        for toko in route:
            total += distance_matrix[pos][toko]
            pos = toko
        total += distance_matrix[pos][0]
    return total

total_jarak = hitung_total_jarak(routes, distance_matrix)
print(f"Total jarak semua truk: {total_jarak:.2f} km")

for i, route in enumerate(routes):
    pos = 0
    jarak = 0
    for toko in route:
        jarak += distance_matrix[pos][toko]
        pos = toko
    jarak += distance_matrix[pos][0]
    print(f"Truk {i+1}: {jarak:.2f} km | {len(route)} toko")

Total jarak semua truk: 277.52 km
Truk 1: 19.33 km | 10 toko
Truk 2: 15.70 km | 10 toko
Truk 3: 16.58 km | 10 toko
Truk 4: 26.71 km | 10 toko
Truk 5: 47.33 km | 10 toko
Truk 6: 20.71 km | 10 toko
Truk 7: 24.92 km | 10 toko
Truk 8: 27.98 km | 10 toko
Truk 9: 31.15 km | 10 toko
Truk 10: 47.10 km | 6 toko


In [11]:
import folium

colors = ['red','blue','green','purple','orange',
          'darkred','darkblue','darkgreen','cadetblue','black']

m = folium.Map(location=[-7.28, 112.73], zoom_start=12)

folium.Marker(
    location=[df_full.iloc[0]['lat'], df_full.iloc[0]['lon']],
    popup='DEPOT',
    icon=folium.Icon(color='red', icon='home')
).add_to(m)

for i, route in enumerate(routes):
    coords = [df_full.iloc[0][['lat','lon']].tolist()]
    for idx in route:
        row = df_full.iloc[idx]
        coords.append([row['lat'], row['lon']])
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=5,
            color=colors[i],
            fill=True,
            popup=f"Truk {i+1} | {row['brand']}"
        ).add_to(m)
    coords.append(df_full.iloc[0][['lat','lon']].tolist())

    folium.PolyLine(coords, color=colors[i], weight=2).add_to(m)

m.save('vrp_routes.html')
print("Map saved!")

Map saved!


![alt text](image-1.png)

In [12]:
from sklearn.cluster import KMeans

coords = df[['lat', 'lon']].values

kmeans = KMeans(n_clusters=10, random_state=42)
df['cluster'] = kmeans.fit_predict(coords)

print(df['cluster'].value_counts().sort_index())

cluster
0     2
1    15
2    10
3    15
4    10
5    11
6     9
7     3
8    14
9     7
Name: count, dtype: int64


In [13]:
pip install ortools

   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
    --------------------------------------- 0.5/23.8 MB 4.2 MB/s eta 0:00:06
   -- ------------------------------------- 1.6/23.8 MB 4.4 MB/s eta 0:00:06
   --- ------------------------------------ 1.8/23.8 MB 3.9 MB/s eta 0:00:06
   --- ------------------------------------ 2.4/23.8 MB 3.2 MB/s eta 0:00:07
   ---- ----------------------------------- 2.9/23.8 MB 2.8 MB/s eta 0:00:08
   ----- ---------------------------------- 3.1/23.8 MB 2.7 MB/s eta 0:00:08
   ----- ---------------------------------- 3.1/23.8 MB 2.7 MB/s eta 0:00:08
   ----- ---------------------------------- 3.1/23.8 MB 2.7 MB/s eta 0:00:08
   ----- ---------------------------------- 3.1/23.8 MB 2.7 MB/s eta 0:00:08
   ----- ---------------------------------- 3.1/23.8 MB 2.7 MB/s eta 0:00:08
   ----- ---------------------------------- 3.1/23.8 MB 2.7 MB/s eta 0:00:08
   ----- ---------------------------------- 3.1/23.8 MB 2.7 MB/s eta 0:00:08
   ---

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.14 requires protobuf<5,>=4.25.3, but you have protobuf 6.33.6 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

In [15]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

def create_data_model():
    data = {}

    data['distance_matrix'] = (distance_matrix * 100).astype(int).tolist()
    data['demands'] = [0] + [1] * len(df)
    data['vehicle_capacities'] = [VEHICLE_CAPACITY] * NUM_VEHICLES
    data['num_vehicles'] = NUM_VEHICLES
    data['depot'] = 0

    return data

data = create_data_model()

print(f"Jumlah node: {len(data['distance_matrix'])}")
print(f"Jumlah kendaraan: {data['num_vehicles']}")
print(f"Kapasitas tiap kendaraan: {data['vehicle_capacities'][0]}")
print(f"Total demand: {sum(data['demands'])}")

Jumlah node: 97
Jumlah kendaraan: 10
Kapasitas tiap kendaraan: 10
Total demand: 96


In [16]:
manager = pywrapcp.RoutingIndexManager(
    len(data['distance_matrix']),
    data['num_vehicles'],
    data['depot']
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data['distance_matrix'][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return data['demands'][from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,
    data['vehicle_capacities'],
    True,
    'Capacity'
)

search_params = pywrapcp.DefaultRoutingSearchParameters()
search_params.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

solution = routing.SolveWithParameters(search_params)

if solution:
    print("Solusi ditemukan!")
else:
    print("Solusi tidak ditemukan.")

Solusi ditemukan!


In [17]:
def print_solution(data, manager, routing, solution):
    total_jarak = 0
    routes_ortools = []

    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route = []
        jarak = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            if node != data['depot']:
                route.append(node)
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            jarak += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)

        jarak_km = jarak / 100
        total_jarak += jarak_km
        routes_ortools.append(route)
        print(f"Truk {vehicle_id+1}: {len(route)} toko | {jarak_km:.2f} km")

    print(f"\nTotal jarak OR-Tools: {total_jarak:.2f} km")
    return routes_ortools

routes_ortools = print_solution(data, manager, routing, solution)

Truk 1: 10 toko | 36.18 km
Truk 2: 10 toko | 28.53 km
Truk 3: 10 toko | 26.57 km
Truk 4: 10 toko | 22.30 km
Truk 5: 10 toko | 21.03 km
Truk 6: 10 toko | 15.52 km
Truk 7: 9 toko | 18.51 km
Truk 8: 10 toko | 25.99 km
Truk 9: 10 toko | 29.37 km
Truk 10: 7 toko | 14.58 km

Total jarak OR-Tools: 238.58 km


In [18]:
m2 = folium.Map(location=[-7.28, 112.73], zoom_start=12)

# Depot
folium.Marker(
    location=[df_full.iloc[0]['lat'], df_full.iloc[0]['lon']],
    popup='DEPOT',
    icon=folium.Icon(color='red', icon='home')
).add_to(m2)

for i, route in enumerate(routes_ortools):
    coords = [df_full.iloc[0][['lat','lon']].tolist()]
    for idx in route:
        row = df_full.iloc[idx]
        coords.append([row['lat'], row['lon']])
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=5,
            color=colors[i],
            fill=True,
            popup=f"Truk {i+1} | {row['brand']}"
        ).add_to(m2)
    coords.append(df_full.iloc[0][['lat','lon']].tolist())
    folium.PolyLine(coords, color=colors[i], weight=2).add_to(m2)

m2.save('vrp_routes_ortools.html')
print("Map saved!")

Map saved!


![alt text](image-2.png)